# Davinci Notebook 9 — Unified Runtime Integration & Benchmarking

A self-contained reference decoder integrates GQA, RoPE, RMSNorm, SwiGLU, hybrid local/global layers, SDPA, and incremental KV cache. Tests are deliberately small. The implementation is educational/reference code; dense-mask prefill is not a long-context kernel.

In [ ]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch:', torch.__version__)


In [ ]:
def rotate_half(x): a,b=x.chunk(2,-1); return torch.cat((-b,a),-1)
def apply_rope(x, positions, theta=10000.):
 d=x.shape[-1]; inv=theta**(-torch.arange(0,d,2,device=x.device,dtype=torch.float32)/d); f=torch.outer(positions.float(),inv); e=torch.cat((f,f),-1); return x*e.cos()[None,None,:,:]+rotate_half(x)*e.sin()[None,None,:,:]
class RMSNorm(nn.Module):
 def __init__(self,d,eps=1e-6): super().__init__(); self.weight=nn.Parameter(torch.ones(d)); self.eps=eps
 def forward(self,x): return (x.float()*torch.rsqrt(x.float().pow(2).mean(-1,keepdim=True)+self.eps)).to(x.dtype)*self.weight
def local_mask(qpos,kpos,window): delta=qpos[:,None]-kpos[None,:]; return (delta>=0)&(delta<window)
class Cache:
 def __init__(self,window=None): self.k=self.v=self.pos=None; self.window=window
 def add(self,k,v,pos):
  if self.k is None: self.k,self.v,self.pos=k,v,pos
  else: self.k=torch.cat([self.k,k],2); self.v=torch.cat([self.v,v],2); self.pos=torch.cat([self.pos,pos])
  if self.window and len(self.pos)>self.window: self.k=self.k[:,:,-self.window:]; self.v=self.v[:,:,-self.window:]; self.pos=self.pos[-self.window:]
  return self.k,self.v,self.pos


In [ ]:
class DavinciReference(nn.Module):
 def __init__(self,c):
  super().__init__(); self.c=c; self.embed=nn.Embedding(c['vocab'],c['hidden']); self.blocks=nn.ModuleList([Block(c,None if (i+1)%c['global_every']==0 else c['window']) for i in range(c['layers'])]); self.norm=RMSNorm(c['hidden']); self.head=nn.Linear(c['hidden'],c['vocab'],False); self.head.weight=self.embed.weight
 def forward(self,ids):
  pos=torch.arange(ids.shape[1],device=ids.device); x=self.embed(ids)
  for blk in self.blocks: x=blk(x,pos)
  return self.head(self.norm(x))
 def decode_one(self,ids,pos,caches):
  x=self.embed(ids)
  for blk,cache in zip(self.blocks,caches): x=blk(x,pos,cache)
  return self.head(self.norm(x))
 def make_caches(self): return [Cache(None if (i+1)%self.c['global_every']==0 else self.c['window']) for i in range(self.c['layers'])]
 def prefill_cache(self,ids):
  caches=self.make_caches(); pos=torch.arange(ids.shape[1],device=ids.device); x=self.embed(ids)
  for blk,cache in zip(self.blocks,caches):
   z=blk.n1(x); b,t,d=z.shape; a=blk.attn; k=a.k(z).view(b,t,a.kvheads,a.hd).transpose(1,2); v=a.v(z).view(b,t,a.kvheads,a.hd).transpose(1,2); k=apply_rope(k,pos); cache.add(k,v,pos)
   x=blk(x,pos)
  return self.head(self.norm(x[:,-1:,:])),caches


## 1. Small model smoke test and cache parity
Tests full-sequence final-token logits against prefill plus one incremental token for global and local attention schedules.

In [ ]:
cfg={'vocab':512,'hidden':128,'layers':4,'heads':4,'kv_heads':2,'head_dim':32,'intermediate':256,'window':8,'global_every':2}
torch.manual_seed(7); model=DavinciReference(cfg).to(DEVICE).eval()
@torch.no_grad()
def test_cache_parity():
 ids=torch.randint(0,cfg['vocab'],(1,13),device=DEVICE); full=model(ids)[:,-1:,:]; pre,caches=model.prefill_cache(ids[:,:-1]); incremental=model.decode_one(ids[:,-1:],torch.tensor([12],device=DEVICE),caches); err=(full-incremental).abs().max().item(); print('max absolute logit difference:',err); assert err<1e-3
test_cache_parity()


## 2. Throughput and memory benchmark (short sequences)
Run this on the actual accelerator. Timings are informative only; compare identical hardware/software conditions.

In [ ]:
@torch.no_grad()
def benchmark(seq=512,repeats=5):
 model.eval(); ids=torch.randint(0,cfg['vocab'],(1,seq),device=DEVICE)
 if DEVICE=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
 t0=time.perf_counter()
 for _ in range(repeats): model(ids)
 if DEVICE=='cuda': torch.cuda.synchronize()
 elapsed=(time.perf_counter()-t0)/repeats; tokens_s=seq/elapsed; peak=torch.cuda.max_memory_allocated()/1024**3 if DEVICE=='cuda' else None
 print({'sequence':seq,'mean_seconds':elapsed,'tokens_per_second':tokens_s,'peak_allocated_GiB':peak})
benchmark()


## 3. Parameter count and scaling checklist
- This notebook's model config is intentionally tiny for tests; replace it with the frozen project config only after validating memory.
- Benchmark prefill at multiple sequence lengths and batch sizes; track peak memory and latency.
- Do not extrapolate a 512-token test to 500K context.
- Production next steps: optimized sliding-window prefill, paged KV cache, batching, robust generation loop, checkpoint compatibility tests, training integration, and multi-GPU context parallelism.
